> ## SOLUTION / ANSWER KEY &mdash; Lab 9.4
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-04-flag-anomalies.ipynb`](../lab-04-flag-anomalies.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 9.4 &mdash; Flag Anomalies

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Flag a margin that declined versus the prior period
- Flag a debt level that spiked beyond a threshold
- Surface what an analyst should look at -- not a decision

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it** (the real grounding / citation / compute logic, or the real `create_agent` wiring), then **Run it** and read the output &mdash; and, for the agent labs, the real **message trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working, grounded insight agent you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The insight agent is a **real** `create_agent` driven by a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")`, key in `.env` as `GROQ_API_KEY`). You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It is **informational only**: it grounds &amp; **cites** every figure, gives **no advice**, and has **no trade tool** (`place_trade` is defined but never bound &mdash; a human analyst decides). A `@tool` must **catch its own errors and return a string** &mdash; a tool that raises can abort the whole run. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing.

**Reference:** [Module 9 slides &mdash; The financial-report insight agent, end to end](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (the Day-5 provider)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-09-04")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is set. Day-5 labs call a REAL hosted model (Groq)."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model. openai/gpt-oss-20b is a reliable tool-caller via create_agent.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY set | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Part of the agent's value is directing a human's attention: **flag** the notable movements &mdash; a
**margin that fell**, **debt that spiked** &mdash; so the analyst spends judgement where it matters (deck
slide 7). A flag is a **signal to look**, never a decision. Thresholds keep it deterministic and
auditable.

In [ ]:
# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

print("this quarter debt:", REPORT["total_debt"]["value"], "M  |  prior:", PRIOR["total_debt"], "M")

## Build it
Complete `analyze_flags`: flag a declining margin and a debt spike (growth over 50%), then run the
cell.

In [ ]:
def analyze_flags(margin_now, margin_prior, debt_growth_pct):
    flags = []
    if margin_now < margin_prior:
        flags.append("margin_down")
    if debt_growth_pct > 50:
        flags.append("debt_spike")
    return flags

try:
    print("margin 7.5 vs 9.1, debt +60% ->", analyze_flags(7.5, 9.1, 60))
    print("margin 9.5 vs 9.1, debt +5%  ->", analyze_flags(9.5, 9.1, 5))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- A flag says **"look here"**, it never says "sell" &mdash; surfacing is not deciding.
- Thresholds are explicit and deterministic, so every flag is **auditable**: you can say exactly why it fired.
- Both conditions can fire at once (margin down **and** debt spike) &mdash; the analyst sees the full picture.

## Your turn (open task &mdash; no grader)
Add a third flag &mdash; e.g. `revenue_down` when YoY growth is negative, or `high_leverage` when
debt-to-revenue crosses a threshold you pick. **What good looks like:** the new flag fires on the right
inputs and stays a signal for a human, not a recommendation.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
def analyze_flags_v2(margin_now, margin_prior, debt_growth_pct, revenue_yoy):
    flags = analyze_flags(margin_now, margin_prior, debt_growth_pct)   # reuse Lab 4's two flags
    if revenue_yoy < 0:                        # a THIRD signal for a human -- still not a recommendation
        flags.append("revenue_down")
    return flags
print("margin 7.5<9.1, debt +60%, rev -4% ->", analyze_flags_v2(7.5, 9.1, 60, -4.0))
print("all healthy                        ->", analyze_flags_v2(9.5, 9.1, 5, 12.0))

---
### Key takeaway
Flags point the analyst at what matters -- a margin that fell as revenue grew, debt that spiked. They surface, they don't decide; the human judges what the flag means.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>